In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType
from delta.tables import DeltaTable

CATALOG = "workspace"
SCHEMA = "oss_ecosystem"

BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_github_events"
SILVER_PUSH_TABLE = f"{CATALOG}.{SCHEMA}.silver_push_events"

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.oss_ecosystem.silver_push_events (
    event_id STRING,
    event_ts TIMESTAMP,
    actor_id BIGINT,
    repo_id BIGINT,
    push_id BIGINT,
    ref STRING,
    before_sha STRING,
    head_sha STRING,
    source_file STRING,
    source_hour TIMESTAMP,
    ingested_at TIMESTAMP
)
USING DELTA
""")

In [0]:
push_bronze_df = (
    spark.table(BRONZE_TABLE)
    .filter(F.col("event_type") == "PushEvent")
)

print("PushEvent rows in Bronze:", push_bronze_df.count())

In [0]:
push_payload_schema = StructType([
    StructField("push_id", LongType(), True),
    StructField("ref", StringType(), True),
    StructField("head", StringType(), True),
    StructField("before", StringType(), True)
])

In [0]:
existing_push_ids = (
    spark.table(SILVER_PUSH_TABLE)
    .select("event_id")
)

new_push_bronze_df = (
    spark.table(BRONZE_TABLE)
    .filter(F.col("event_type") == "PushEvent")
    .join(
        existing_push_ids,
        on="event_id",
        how="left_anti"
    )
)

print(
    "New PushEvents to transform:",
    new_push_bronze_df.count()
)

In [0]:
silver_push_df = (
    new_push_bronze_df
    .withColumn(
        "payload_parsed",
        F.from_json(
            F.col("payload_json"),
            push_payload_schema
        )
    )
    .select(
        "event_id",

        F.to_timestamp("created_at")
            .alias("event_ts"),

        F.get_json_object("actor_json", "$.id")
            .cast("long")
            .alias("actor_id"),

        F.get_json_object("repo_json", "$.id")
            .cast("long")
            .alias("repo_id"),

        F.col("payload_parsed.push_id")
            .alias("push_id"),

        F.col("payload_parsed.ref")
            .alias("ref"),

        F.col("payload_parsed.before")
            .alias("before_sha"),

        F.col("payload_parsed.head")
            .alias("head_sha"),

        "source_file",
        "source_hour",
        "ingested_at"
    )
)

In [0]:
print("Silver Push rows:", silver_push_df.count())

silver_push_df.select(
    F.sum(F.col("event_id").isNull().cast("int"))
        .alias("null_event_id"),

    F.sum(F.col("actor_id").isNull().cast("int"))
        .alias("null_actor_id"),

    F.sum(F.col("repo_id").isNull().cast("int"))
        .alias("null_repo_id"),

    F.sum(F.col("push_id").isNull().cast("int"))
        .alias("null_push_id"),

    F.sum(F.col("ref").isNull().cast("int"))
        .alias("null_ref"),

    F.sum(F.col("head_sha").isNull().cast("int"))
        .alias("null_head_sha"),

    F.sum(F.col("before_sha").isNull().cast("int"))
        .alias("null_before_sha")
).show()

In [0]:
duplicate_count = (
    silver_push_df
    .groupBy("event_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Duplicate PushEvent IDs:", duplicate_count)

In [0]:
if not spark.catalog.tableExists(SILVER_PUSH_TABLE):
    (
        silver_push_df.limit(0)
        .write
        .format("delta")
        .saveAsTable(SILVER_PUSH_TABLE)
    )

silver_push_delta = DeltaTable.forName(
    spark,
    SILVER_PUSH_TABLE
)

(
    silver_push_delta.alias("t")
    .merge(
        silver_push_df.alias("s"),
        "t.event_id = s.event_id"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

print("Silver Push MERGE completed.")

In [0]:
print(
    "Silver Push table rows:",
    spark.table(SILVER_PUSH_TABLE).count()
)

print(
    "Duplicate PushEvent IDs:",
    spark.table(SILVER_PUSH_TABLE)
        .groupBy("event_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
)